# Sample QC of a cohort: relatedness, genetic sex and a trio check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fbcouto/multimerge-genomics-research/blob/main/notebooks/human/cohort_sample_qc.ipynb)

> **Notice.** Experimental software, for teaching and research only: not a medical diagnosis, not a paternity test with legal value and not advice on any treatment or medicine. The authors take no responsibility for decisions made on the basis of these results; a finding must be confirmed by an accredited laboratory and interpreted by a qualified professional. Analyze only genomes of people who agreed to it.

**The problem.** Before an association study, a family analysis or a joint call set, check that the samples
are who the sample sheet says they are. Swapped or duplicated tubes, undeclared relatives (they inflate
associations), a declared parent who is not one, or a genetic sex that differs from the declared one (often
the sign of a swap) must be found before the analysis, not after. This is the identity part of a standard
sample QC, as done with KING or plink.

**What you need.** One VCF per person (or one multi-sample VCF), ideally with each person's callable
regions (BED), and the sample sheet: declared sex, father and mother.

**What you get.** The kinship of every pair and the inferred relationship, compared with the declared one;
the genetic sex compared with the declared one; the Mendelian check of each trio; and a QC verdict per person.

In [ ]:
# Setup. Runs on Google Colab or on your own computer: on Colab, the first run installs MMGEN from the GitHub
# release (about a minute). Tables use pandas and plots matplotlib when installed (Colab has both); without
# them the notebook prints plain text and skips the plots.
import platform, subprocess, sys
from pathlib import Path

RELEASE = "https://github.com/fbcouto/multimerge-genomics-research/releases/download/v2.4.0"
try:
    import multimerge_genomics as mmg
except ImportError:
    if platform.system() != "Linux":
        print("MMGEN is not installed in this Python. Download the wheel for your system from\n"
              "  https://github.com/fbcouto/multimerge-genomics-research/releases/tag/v2.4.0\n"
              "install it with:  python -m pip install <the .whl file>\n"
              "and run this cell again.")
        raise
    print("Installing MMGEN (about a minute)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{RELEASE}/multimerge_genomics-2.4.0-cp310-abi3-manylinux_2_28_x86_64.whl"], check=True)
    import multimerge_genomics as mmg

IN_NOTEBOOK = "ipykernel" in sys.modules
try:
    from IPython.display import display, Markdown
except ImportError:                        # plain Python: print instead
    display, Markdown = print, str
try:
    import pandas as pd
    pd.set_option("display.width", 160)
    pd.set_option("display.max_columns", 30)
except ImportError:
    pd = None
try:
    import matplotlib
    if not IN_NOTEBOOK:
        matplotlib.use("Agg")              # plain Python: save the figures, open no window
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

# Chart colors (a palette checked for color-blind readers): two series, a two-sided scale, neutral ink
BLUE, ORANGE, RED, GRAY, INK = "#2a78d6", "#eb6834", "#e34948", "#a3a29c", "#52514e"


def show(rows, n=25):
    """Shows a list of dictionaries as a table (pandas when available, plain text otherwise)."""
    rows = list(rows)
    if pd is not None:
        display(pd.DataFrame(rows[:n]))
    else:
        for row in rows[:n]:
            print("  ".join(f"{k}: {v}" for k, v in row.items()))
    if len(rows) > n:
        print(f"... and {len(rows) - n} more rows")


def tidy(ax):
    """Recessive axes: no top or right frame, a light grid behind the data."""
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e4e3df", linewidth=0.8)
    ax.set_axisbelow(True)


def save_figure(fig, path):
    """Saves a figure; shows it in a notebook, closes it in plain Python."""
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("Figure saved to", path)
    if IN_NOTEBOOK:
        plt.show()
    else:
        plt.close(fig)


print(mmg.engine_info())

## Data

By default, from the human demo pack (downloaded once, about 45 MB): the seven Genome in a Bottle (NIST)
reference genomes, public data from volunteers who consented to open sharing: HG001 (NA12878), the
Ashkenazi Jewish trio HG002 (son), HG003 (father), HG004 (mother) and the Han Chinese trio HG005 (son),
HG006 (father), HG007 (mother). Their benchmark calls v4.2.1 are cut to chromosome 22 and some genes; their
callable regions come with them; chrX of HG001 (v3.3.2) and chrX/chrY of HG002 (v5.0q) are separate files,
because v4.2.1 has no sex chromosomes. The 1000 Genomes allele frequencies are used for the paternity index.

To use your own cohort, fill `MY_COHORT` (one entry per person) and, optionally, `MY_ALLELE_FREQUENCIES`.

In [ ]:
# Your cohort (None = the demo data): one entry per person
#   "vcf": the person's VCF; "bed": the callable regions (None: a site absent from the VCF counts as homozygous
#   reference everywhere, as for whole-genome calls); "sex": declared "F" or "M"; "father"/"mother": the ids of
#   the declared parents in this table, or None; "sex_vcf" (optional): a VCF with chrX/chrY if "vcf" has none.
MY_COHORT = None             # e.g. {"S1": {"vcf": "S1.vcf.gz", "bed": None, "sex": "F", "father": None, "mother": None}, ...}
MY_ALLELE_FREQUENCIES = None # a sorted sites VCF with population allele frequencies in INFO (optional)
AF_FIELD = "AF"              # its INFO field (e.g. "EUR_AF" for a European cohort)

import urllib.request, zipfile

DATA = Path("data")                          # the demo files are extracted here
OUT = Path("results") / "cohort_sample_qc"         # everything this notebook writes
OUT.mkdir(parents=True, exist_ok=True)


def demo_file(pack, member):
    """One file of a demo pack: downloads the pack once into the working folder and extracts only that file."""
    target = DATA / member
    if not target.exists():
        if not Path(pack).exists():
            print(f"Downloading {pack} from the MMGEN release...")
            urllib.request.urlretrieve(f"{RELEASE}/{pack}", pack + ".part")
            Path(pack + ".part").replace(pack)
        with zipfile.ZipFile(pack) as z:
            z.extract(member, DATA)
    return str(target)


USING_DEMO = MY_COHORT is None
if USING_DEMO:
    giab = lambda person, ext: demo_file("mmgen_demo_human_v3.zip", f"human/giab/{person}.{ext}")
    sheet = {"HG001": ("F", None, None), "HG002": ("M", "HG003", "HG004"), "HG003": ("M", None, None),
             "HG004": ("F", None, None), "HG005": ("M", "HG006", "HG007"), "HG006": ("M", None, None),
             "HG007": ("F", None, None)}
    COHORT = {p: {"vcf": giab(p, "vcf.gz"), "bed": giab(p, "callable.bed"), "sex": s, "father": f, "mother": m}
              for p, (s, f, m) in sheet.items()}
    COHORT["HG001"]["sex_vcf"] = giab("HG001", "chrX.vcf.gz")
    COHORT["HG002"]["sex_vcf"] = giab("HG002", "chrXY.vcf.gz")
    AF_VCF = demo_file("mmgen_demo_human_v3.zip", "human/af_1000g_demo.vcf.gz")
else:
    COHORT, AF_VCF = MY_COHORT, MY_ALLELE_FREQUENCIES
show({"person": p, "declared sex": c["sex"], "father": c.get("father") or "-", "mother": c.get("mother") or "-",
      "VCF": Path(c["vcf"]).name} for p, c in COHORT.items())

## Step 1. Relatedness of every pair

The KING-robust kinship coefficient (as `plink2 --make-king-table`) uses the SNVs where at least one of the
two people is heterozygous, so it does not need allele frequencies and works across ancestries within
limits. Expected values: about 0.5 for duplicates or identical twins, 0.25 for parent and child or full
siblings (told apart by IBS0, the share of sites where they have no allele in common: near 0 for parent
and child), 0.125 for second degree and 0.0625 for third degree. Between people of different ancestries
the kinship comes out negative; that is a known property of the estimator, not an error.

A site absent from a person's VCF can mean "homozygous reference" or "not callable": the callable BED
says which, and without it absence is read as reference.

After the table, a heatmap with one cell per pair: dark cells are close relatives; negative values (different
ancestries) are colored as 0.

In [ ]:
people = list(COHORT)
beds = [COHORT[p].get("bed") for p in people]
rel = mmg.relatedness([COHORT[p]["vcf"] for p in people], callable_beds=beds if all(beds) else None, labels=people)
print(f"{len(people)} people, {len(rel['pairs'])} pairs, {rel['snv_sites']:,} biallelic autosomal SNV sites on {rel['autosomes']} autosome(s)")
print("Kinship thresholds used:", {k: round(v, 4) for k, v in rel["thresholds"].items()})


def declared(a, b):
    A, B = COHORT[a], COHORT[b]
    if b in (A.get("father"), A.get("mother")) or a in (B.get("father"), B.get("mother")):
        return "parent_offspring"
    if A.get("father") and (A.get("father"), A.get("mother")) == (B.get("father"), B.get("mother")):
        return "full_siblings"
    return "unrelated"


pairs = []
for p in rel["pairs"]:
    expected = declared(p["a"], p["b"])
    found = p["relationship"]
    status = "n/a" if found == "insufficient_data" else ("OK" if found == expected else "MISMATCH")
    pairs.append({"person A": p["a"], "person B": p["b"], "kinship": round(p["kinship"], 4),
                  "IBS0/HETHET": None if p["ibs0_ratio"] is None else round(p["ibs0_ratio"], 4),
                  "inferred": found, "declared": expected, "status": status})
pairs.sort(key=lambda r: -r["kinship"])
show([r for r in pairs if r["inferred"] != "unrelated" or r["declared"] != "unrelated" or r["status"] != "OK"], n=50)
print(f"{sum(r['status'] == 'MISMATCH' for r in pairs)} pair(s) where the inferred relationship differs from the sheet; "
      f"the other {sum(r['status'] == 'OK' and r['declared'] == 'unrelated' for r in pairs)} pairs are unrelated, as declared")
if rel["autosomes"] < 10:
    print(f"Only {rel['autosomes']} autosome(s): the degree of distant relatives is uncertain; a whole genome separates them better.")

if plt is not None:
    from matplotlib.colors import LinearSegmentedColormap

    ramp = LinearSegmentedColormap.from_list("kinship", ["#ffffff", "#cde2fb", "#6da7ec", "#2a78d6", "#104281"])
    index = {p: i for i, p in enumerate(people)}
    kinship = [[float("nan")] * len(people) for _ in people]       # the diagonal stays blank
    for p in rel["pairs"]:
        i, j = index[p["a"]], index[p["b"]]
        kinship[i][j] = kinship[j][i] = p["kinship"]
    matrix = [[max(k, 0.0) if k == k else k for k in row] for row in kinship]   # color: negative as 0
    fig, ax = plt.subplots(figsize=(1 + 0.6 * len(people), 0.6 * len(people)))
    image = ax.imshow(matrix, cmap=ramp, vmin=0, vmax=0.5)
    ax.set_xticks(range(len(people)), people, rotation=45, ha="right")
    ax.set_yticks(range(len(people)), people)
    if len(people) <= 15:
        for i in range(len(people)):
            for j in range(len(people)):
                if i != j:
                    ax.text(j, i, f"{kinship[i][j]:.2f}", ha="center", va="center", fontsize=7,
                            color="white" if matrix[i][j] > 0.3 else INK)
    fig.colorbar(image, ax=ax, label="KING kinship (color: negative as 0)", shrink=0.8)
    ax.set_title("Kinship of every pair")
    save_figure(fig, OUT / "kinship.png")

## Step 2. Genetic sex

Two X chromosomes give heterozygous calls on chrX (outside the pseudoautosomal regions, which X and Y
share) at about the rate of the autosomes; one X gives almost none, and a Y gives calls on chrY. A file
without chrX cannot tell (the demo's v4.2.1 cuts have none, so only HG001 and HG002 are tested, from their
separate chrX/chrY files). A mismatch with the sheet is the commonest sign of a sample swap; X0 or XXY
patterns are flagged.

In [ ]:
expected_sex = {"F": "XX", "M": "XY"}
sex_rows = []
for p, c in COHORT.items():
    s = mmg.genetic_sex(c.get("sex_vcf") or c["vcf"])
    if s["verdict"] == "undetermined":
        status = "no X data" if "no_x_data" in s["flags"] else "undetermined"
    else:
        status = "OK" if s["verdict"] == expected_sex.get(c["sex"]) else "MISMATCH"
    sex_rows.append({"person": p, "declared": c["sex"], "genetic": s["verdict"],
                     "chrX het / autosomes": None if s["x_het_ratio"] is None else round(s["x_het_ratio"], 2),
                     "chrY calls": s["y_calls"], "flags": ", ".join(s["flags"]) or "-", "status": status})
show(sex_rows)

## Step 3. Trios: Mendelian check and parentage

For each person with both parents in the cohort: at every site where all three have a genotype, the child's
alleles must come one from each parent. The Mendelian error rate is a genotyping-quality measure (well below
0.1% for good calls); a rate of several percent means a wrong parent or a sample swap. Opposite homozygotes
between a parent and the child exclude parentage, unless they are as rare as genotyping errors. De novo
candidates (child heterozygous, both parents homozygous reference) are mostly genotyping errors: real ones
are confirmed in the laboratory. The combined paternity index treats sites as independent, so it is far
larger than a forensic marker panel would give: read it as teaching material, not as evidence.

At the end, with the demo data, the same check with a wrong father (HG006, of the other trio, tested as the
father of HG002) shows what a sample swap looks like.

In [ ]:
trio_rows, trios = [], {}
for child, c in COHORT.items():
    father, mother = c.get("father"), c.get("mother")
    if father not in COHORT or mother not in COHORT:
        continue
    beds = {f"{role}_bed": COHORT[p].get("bed") for role, p in (("child", child), ("father", father), ("mother", mother))}
    t = mmg.trio_analysis(c["vcf"], COHORT[father]["vcf"], mother=COHORT[mother]["vcf"],
                          **{k: v for k, v in beds.items() if v},
                          allele_frequencies=AF_VCF, af_field=AF_FIELD,
                          de_novo_output=str(OUT / f"{child}.de_novo.vcf"))
    trios[child] = t
    trio_rows.append({"child": child, "father": father, "mother": mother, "sites checked": t["sites_checked"],
                      "Mendelian errors": t["errors"], "error rate %": round(100 * t["error_rate"], 4),
                      "de novo candidates": t["de_novo"],
                      "father exclusions": f"{t['father_exclusions']} of {t['father_informative']:,}",
                      "mother exclusions": f"{t['mother_exclusions']} of {t['mother_informative']:,}",
                      "verdict": t["verdict"],
                      "log10 paternity index": None if t["log10_cpi"] is None else round(t["log10_cpi"])})
show(trio_rows)
for child, t in trios.items():
    print(f"{child}: {t['reason']}")

if USING_DEMO:
    wrong = mmg.trio_analysis(COHORT["HG002"]["vcf"], COHORT["HG006"]["vcf"], mother=COHORT["HG004"]["vcf"],
                              child_bed=COHORT["HG002"]["bed"], father_bed=COHORT["HG006"]["bed"],
                              mother_bed=COHORT["HG004"]["bed"], allele_frequencies=AF_VCF, af_field=AF_FIELD)
    print(f"HG002 with HG006 as father: Mendelian error rate {100 * wrong['error_rate']:.2f}% "
          f"(true trio: {100 * trios['HG002']['error_rate']:.4f}%), father exclusions "
          f"{wrong['father_exclusions']:,} of {wrong['father_informative']:,}; verdict: {wrong['verdict']}")

## Step 4. QC verdict per person

A person passes when the declared relationships, the declared sex (when it can be tested) and, for a
child, the Mendelian check all agree. A person to review is not necessarily wrong: look at the evidence
above before excluding anyone, and ask the laboratory for the sample's history.

In [ ]:
verdicts = []
for p, c in COHORT.items():
    problems = [f"relationship with {r['person B'] if r['person A'] == p else r['person A']}"
                for r in pairs if r["status"] == "MISMATCH" and p in (r["person A"], r["person B"])]
    sex = next(r for r in sex_rows if r["person"] == p)
    if sex["status"] == "MISMATCH":
        problems.append(f"sex: declared {c['sex']}, genetic {sex['genetic']}")
    if p in trios and (trios[p]["error_rate"] > 0.001 or trios[p]["verdict"] != "not_excluded"):
        problems.append(f"trio: {100 * trios[p]['error_rate']:.3f}% Mendelian errors, {trios[p]['verdict']}")
    verdicts.append({"person": p, "sex tested": sex["status"] != "no X data", "QC": "REVIEW" if problems else "PASS",
                     "why": "; ".join(problems) or "-"})
show(verdicts)
display(Markdown(f"**{sum(v['QC'] == 'PASS' for v in verdicts)} of {len(verdicts)} people pass.**"))

## Check the result

In [ ]:
def check(condition, text):
    print(("OK     " if condition else "CHECK  ") + text)


check(rel["snv_sites"] > 10_000, f"enough SNV sites for kinship ({rel['snv_sites']:,})")
check(all(p["sites"] > 0.5 * rel["snv_sites"] for p in rel["pairs"]), "every pair shares most of the sites")
check(not any(r["inferred"] == "duplicate_or_mz_twin" for r in pairs), "no duplicate sample")
check(all(t["error_rate"] < 0.001 for t in trios.values()), "every trio has under 0.1% Mendelian errors")
if USING_DEMO:
    parent_child = {frozenset((r["person A"], r["person B"])) for r in pairs if r["inferred"] == "parent_offspring"}
    assert parent_child == {frozenset(x) for x in (("HG002", "HG003"), ("HG002", "HG004"), ("HG005", "HG006"), ("HG005", "HG007"))}
    assert {r["person"]: r["genetic"] for r in sex_rows if r["status"] != "no X data"} == {"HG001": "XX", "HG002": "XY"}
    assert all(t["verdict"] == "not_excluded" for t in trios.values()) and wrong["verdict"] == "excluded"
    print("Demo expectations met: four parent-child pairs, HG001 XX and HG002 XY, both trios consistent, the wrong father excluded.")

## Methods paragraph

A starting text for the methods section of a paper or report, filled in with the version and the parameters
this run used. Edit it to your study, and cite the methods MMGEN reproduces (see References).

In [ ]:
import textwrap
VERSION = mmg.engine_info()["version"]
methods = f"""Sample identity and relationships were checked with MMGEN version {VERSION} (Couto F. MMGEN: MultiMerge Genomics (version {VERSION}). Zenodo. https://doi.org/10.5281/zenodo.20417492).
Pairwise KING-robust kinship and IBS0 were computed over {rel['snv_sites']:,} biallelic autosomal SNV sites
(relatedness{'; a site absent from a VCF was counted as homozygous reference only within the callable regions of that sample' if all(beds) else ''})
and compared with the declared pedigree. Genetic sex was inferred from chrX heterozygosity outside the pseudoautosomal
regions and from chrY calls (genetic_sex). Each trio was checked for Mendelian consistency, parental exclusions and
de novo candidates (trio_analysis{', with ' + AF_FIELD + ' allele frequencies for the combined paternity index' if AF_VCF else ''}).
Experimental software, for teaching and research only: not a medical diagnosis, not a paternity test with legal value and not advice on any treatment or medicine. The authors take no responsibility for decisions made on the basis of these results; a finding must be confirmed by an accredited laboratory and interpreted by a qualified professional. Analyze only genomes of people who agreed to it."""
print(textwrap.fill(" ".join(methods.split()), 110))

## Adapt it to your data

* One multi-sample VCF: call `mmg.relatedness("cohort.vcf.gz", samples=[...])` instead of a list of files,
  and give `child_sample=`, `father_sample=`, `mother_sample=` to `trio_analysis`.
* The VCFs must be sorted in natural chromosome order (1..22, X, Y): `mmg.sort_genomic(..., chrom_order="natural")`.
* Split multiallelic records first (`mmg.vcf_normalize`): they are left out of these checks.
* Exome or panel data: always give the callable BED; without it, absent sites outside the target are read
  as homozygous reference and the kinship is biased.
* Choose `AF_FIELD` matching the cohort's ancestry (`EUR_AF`, `EAS_AF`...) for the paternity index.

## References

* Manichaikul A. et al. (2010) Robust relationship inference in genome-wide association studies.
  *Bioinformatics* 26:2867 (KING).
* Chang C.C. et al. (2015) Second-generation PLINK: rising to the challenge of larger and richer datasets.
  *GigaScience* 4:7.
* Anderson C.A. et al. (2010) Data quality control in genetic case-control association studies.
  *Nature Protocols* 5:1564.
* Zook J.M. et al. (2016) Extensive sequencing of seven human genomes to characterize benchmark reference
  materials. *Scientific Data* 3:160025 (the GIAB genomes).
* Wagner J. et al. (2022) Benchmarking challenging small variants with linked and long reads. *Cell Genomics* 2:100128.
* Lowy-Gallego E. et al. (2019) Variant calling on the GRCh38 assembly with the data from phase three of the
  1000 Genomes Project. *Wellcome Open Research* 4:50.